In [ ]:
import numpy as np
import polars as pl


data = pl.read_parquet("../data/1m/1m.parquet").lazy()

In [ ]:
pairs = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "DOGEUSDT"]

In [ ]:
def backtest_pairs_trading(
    data: pl.LazyFrame,
    symbol1: str,
    symbol2: str,
    start_date,
    end_date,
    entry_z: float = 2.5,
    exit_z: float = 0.6,
    fee_bps: float = 10,
    window_size: int = 120,
    print_metrics: bool = True,
):
    """
    Backtest a pairs trading strategy on two symbols.

    Parameters:
    -----------
    data : polars.LazyFrame
        Price data with columns: symbol, open_time, close
    symbol1 : str
        First symbol (e.g., 'BTCUSDT')
    symbol2 : str
        Second symbol (e.g., 'ETHUSDT')
    start_date : datetime
        Start date for backtest
    end_date : datetime
        End date for backtest
    entry_z : float
        Z-score threshold for entry (default: 2.5)
    exit_z : float
        Z-score threshold for exit (default: 0.6)
    fee_bps : float
        Fee in basis points per leg (default: 0.2)
    window_size : int
        Rolling window size for calculations (default: 120)
    print_metrics : bool
        Whether to print performance metrics (default: True)

    Returns:
    --------
    dict with keys: 'backtest', 'metrics', 'data'
    """

    fee_rate = fee_bps / 1e4

    # Prepare data
    data_combined = (
        data.filter(pl.col("symbol") == symbol1)
        .select(
            "open_time",
            pl.col("close").alias(f"{symbol1}_price"),
            pl.col("close").log().alias("S1"),
        )
        .join(
            data.filter(pl.col("symbol") == symbol2).select(
                "open_time",
                pl.col("close").alias(f"{symbol2}_price"),
                pl.col("close").log().alias("S2"),
            ),
            on="open_time",
        )
    ).filter((pl.col("open_time") >= start_date) & (pl.col("open_time") < end_date))

    # Backtest
    bt = (
        data_combined.sort("open_time")
        .with_columns(
            pl.rolling_cov("S1", "S2", window_size=window_size).alias("rolling_cov"),
            pl.col("S2").rolling_var(window_size=window_size).alias("rolling_var_s2"),
        )
        .with_columns(
            (pl.col("rolling_cov") / pl.col("rolling_var_s2")).alias("rolling_beta"),
        )
        .with_columns(
            (
                pl.col("S1").rolling_mean(window_size=window_size)
                - pl.col("rolling_beta")
                * pl.col("S2").rolling_mean(window_size=window_size)
            ).alias("rolling_alpha"),
        )
        .with_columns(
            (
                pl.col("S1")
                - (pl.col("rolling_alpha") + pl.col("rolling_beta") * pl.col("S2"))
            ).alias("rolling_residual"),
        )
        .with_columns(
            (
                (
                    pl.col("rolling_residual")
                    - pl.col("rolling_residual").rolling_mean(window_size=window_size)
                )
                / pl.col("rolling_residual").rolling_std(window_size=window_size)
            ).alias("rolling_zscore"),
        )
        .with_columns(
            pl.col(f"{symbol1}_price").pct_change().alias("r_s1"),
            pl.col(f"{symbol2}_price").pct_change().alias("r_s2"),
            pl.col("rolling_beta").shift(1).alias("beta_lag"),
            pl.col("rolling_zscore").shift(1).alias("z_lag"),
        )
        .filter(pl.col("r_s2").is_not_null())
        .with_columns(
            pl.when(pl.col("z_lag") >= entry_z)
            .then(-1)
            .when(pl.col("z_lag") <= -entry_z)
            .then(1)
            .when(pl.col("z_lag").abs() <= exit_z)
            .then(0)
            .otherwise(None)
            .forward_fill()
            .alias("pos")
        )
        .with_columns(
            (pl.col("r_s1") - pl.col("beta_lag") * pl.col("r_s2")).alias("r_spread"),
            pl.col("pos").shift(1).fill_null(0).alias("pos_prev"),
        )
        .with_columns(
            (pl.col("pos") * pl.col("r_spread")).alias("pnl_gross"),
            (
                (pl.col("pos") - pl.col("pos_prev")).abs()
                * (1 + pl.col("beta_lag").abs())
            ).alias("turnover_units"),
        )
        .with_columns((fee_rate * pl.col("turnover_units")).alias("fees"))
        .with_columns((pl.col("pnl_gross") - pl.col("fees")).alias("pnl_net"))
        .with_columns(
            (1 + pl.col("pnl_net")).cum_prod().alias("equity"),
            pl.col("pnl_net").cum_sum().alias("cum_return"),
        )
        .drop_nulls()
    )

    out = bt.collect()

    # Calculate metrics
    ann_factor = 365 * 24 * 60
    mean_pnl = out["pnl_net"].mean()
    std_pnl = out["pnl_net"].std()
    sharpe_ratio = (
        (mean_pnl / std_pnl) * np.sqrt(ann_factor)
        if std_pnl and std_pnl > 0
        else float("nan")
    )
    trades_count = int((out["pos"] != out["pos_prev"]).sum() // 2)

    equity_arr = out["equity"].to_numpy()
    roll_max = np.maximum.accumulate(equity_arr)
    dd = (equity_arr / roll_max) - 1.0
    max_drawdown = dd.min()

    metrics = {
        "symbol1": symbol1,
        "symbol2": symbol2,
        "bars": len(out),
        "trades": trades_count,
        "mean_minute_pnl": float(mean_pnl),
        "std_minute_pnl": float(std_pnl),
        "sharpe_annualized": float(sharpe_ratio),
        "max_drawdown": float(max_drawdown),
        "final_equity": float(equity_arr[-1]) if len(equity_arr) else None,
        "total_return": float(out["cum_return"][-1]) if len(out) else None,
    }

    if print_metrics:
        print(metrics)

    return {"backtest": out, "metrics": metrics, "data": data_combined.collect()}

In [ ]:
from datetime import datetime
from itertools import product

# Define parameter ranges
entry_z_range = [2.0, 2.5, 3.0]
exit_z_range = [0.3, 0.5, 0.6, 0.8]
window_size_range = [60, 120, 180, 240, 480, 720, 960, 1200, 1440, 2160, 2880, 4320]

# Define date range for backtest
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 12, 31)

# Store results
results = []

# Grid search over all parameter combinations
for symbol1, symbol2 in [
    (pairs[i], pairs[j]) for i in range(len(pairs)) for j in range(i + 1, len(pairs))
]:
    for entry_z, exit_z, window_size in product(
        entry_z_range, exit_z_range, window_size_range
    ):
        try:
            result = backtest_pairs_trading(
                data,
                symbol1=symbol1,
                symbol2=symbol2,
                start_date=start_date,
                end_date=end_date,
                entry_z=entry_z,
                exit_z=exit_z,
                window_size=window_size,
                print_metrics=False,
                fee_bps=3.0,
            )

            metrics = result["metrics"]
            metrics["entry_z"] = entry_z
            metrics["exit_z"] = exit_z
            metrics["window_size"] = window_size
            results.append(metrics)
        except Exception as e:
            pass

# Convert to DataFrame and sort by Sharpe ratio
results_df = pl.DataFrame(results).sort("sharpe_annualized", descending=True)
print(results_df.head(10))

In [ ]:
pl.DataFrame(results).sort("sharpe_annualized", descending=True).drop_nans().head(10)